# CA22 — Demo: debug training loop

This notebook demonstrates a short, import-safe debug training run for CA22. It uses the `debug.yaml` config and saves figures to `../pictures/` (unexecuted).


In [ ]:
# Imports & setup
import sys
from pathlib import Path

# make src importable when the notebook is run from notebooks/
sys.path.insert(0, str(Path("..").resolve()))
from src.config import ExperimentConfig
from src.utils import set_seed
import torch
import matplotlib.pyplot as plt

# load debug config
cfg = ExperimentConfig.from_yaml(str(Path("..") / "configs" / "debug.yaml"))
set_seed(cfg.seed)
device = torch.device(cfg.device)

In [ ]:
# Data, models and optimizers
from src.data import SyntheticDataset
from src.model import PolicyNet, ValueNet
from src.losses import policy_gradient_loss, value_loss, LagrangianLoss
from src.utils import update_lagrange

# small synthetic dataset for debug runs
ds = SyntheticDataset(num_episodes=30, obs_dim=cfg.obs_dim, horizon=8, seed=cfg.seed)

policy = PolicyNet(cfg.obs_dim, cfg.action_dim, cfg.hidden_size).to(device)
value = ValueNet(cfg.obs_dim, cfg.hidden_size).to(device)
opt_policy = torch.optim.Adam(policy.parameters(), lr=cfg.lr)
opt_value = torch.optim.Adam(value.parameters(), lr=cfg.lr)
mu = 0.0

In [ ]:
# Short training loop (debug)
loss_history = {"pg": [], "v": [], "combined": [], "mu": []}
for epoch in range(cfg.epochs):
    states, actions, rewards, constraints = ds.sample_batch(batch_size=cfg.batch_size)
    obs = torch.from_numpy(states).to(device)
    actions_t = torch.from_numpy(actions).to(device)
    rewards_t = torch.from_numpy(rewards).to(device)
    constraints_t = torch.from_numpy(constraints).to(device)

    logits = policy(obs)
    dist = torch.distributions.Categorical(logits=logits)
    logp = dist.log_prob(actions_t)
    v_pred = value(obs)
    returns = rewards_t  # no discount for debug
    advantages = returns - v_pred.detach()

    pg = policy_gradient_loss(logp, advantages)
    v_l = value_loss(v_pred, returns)
    lag = LagrangianLoss(mu=mu, constraint_threshold=cfg.constraint_threshold)
    combined = lag(pg, constraints_t)

    opt_policy.zero_grad()
    combined.backward()
    opt_policy.step()

    opt_value.zero_grad()
    v_l.backward()
    opt_value.step()

    # update mu using batch mean constraint
    mean_c = float(constraints_t.mean().item())
    mu = update_lagrange(
        mu, mean_c, cfg.constraint_threshold, cfg.lagrange_lr, cfg.max_mu
    )

    loss_history["pg"].append(pg.item())
    loss_history["v"].append(v_l.item())
    loss_history["combined"].append(combined.item())
    loss_history["mu"].append(mu)

In [ ]:
# Plot and save figures
Path("../pictures").mkdir(parents=True, exist_ok=True)
plt.figure()
plt.plot(loss_history["pg"], label="pg")
plt.plot(loss_history["v"], label="value")
plt.plot(loss_history["combined"], label="combined")
plt.legend()
plt.xlabel("step")
plt.ylabel("loss")
plt.title("Loss curves")
plt.savefig("../pictures/fig_01_loss.png", dpi=200)

plt.figure()
plt.plot(loss_history["mu"])
plt.xlabel("step")
plt.ylabel("mu")
plt.title("Lagrange multiplier")
plt.savefig("../pictures/fig_02_mu.png", dpi=200)